# Notebook 01 — Understanding OmniDocBench

> **阶段**：Stage 1 Use · **预计时间**：40–60 分钟 · **平台**：Kaggle Notebook

| 资源 | 估算 | 说明 |
| --- | --- | --- |
| GPU VRAM | 不需要 GPU | 纯数据理解 |
| GPU 时间 | 0 | CPU 即可 |
| 磁盘 | 实测 1.38 GB 图片 + 42 MB 标注 | 下载 images/ 与 OmniDocBench.json |
| Internet | 需要（首次） | Hugging Face 下载 |


# Learning Objectives

完成本 Notebook 后，你应该能够：

- 说明 OmniDocBench 的数据版本、规模与文件结构；
- 读懂标注 schema（layout_dets / page_info / relation）；
- 可视化不同文档类型的标注并解释差异；
- 用统计数字说明「Benchmark 不是单一数字，而是一组难度不同的数据」。


# Why This Matters

在训练任何模型之前先理解 Benchmark：它测什么、数据长什么样、哪些页面难。否则后面得到的所有分数都无法解释。本 Notebook 不训练、不评测，只建立数据直觉。


# Concepts

- **OmniDocBench**：面向真实文档的端到端文档解析 Benchmark（CVPR 2025，arXiv:2412.07626）；
- **官方数据**：HF `opendatalab/OmniDocBench`，锁定 revision `aa1ee96d…`；
- **版本**：1651 页全量（README 称 v1.6），其中 296 页为 2026-04 新增的困难子集（equation_hard 100 / layout_hard 99 / table_hard 97），其余 1355 页来自 v1.5；
- **评测维度**：end-to-end（end2end / md2md）、layout、table、formula、text OCR；
- **使用限制**：仅研究用途、不可商用。


## Step 1 — 定位或下载数据

先尝试自动定位（Kaggle Dataset 挂载或本地 data/ 目录）；找不到再下载。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data

try:
    data_root = data.find_dataset_root()
    print('dataset root:', data_root)
except FileNotFoundError as e:
    print(e)
    print('请任选一种方式获取数据：')
    print('A. Kaggle Add Input 添加官方 OmniDocBench 数据集（若有官方镜像）；')
    print('B. 执行下一格从 Hugging Face 下载（约 1-2 GB，仅研究用途）。')


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data
# 仅在数据缺失时运行：
# data_root = data.download_dataset()
# print('downloaded to:', data_root)


## Step 2 — 读懂标注 schema

每页是一个 JSON 对象：`layout_dets[]`（块级标注）+ `page_info`（页面元数据）+ `extra.relation[]`（图文关系与段落截断关系）。字段含义见官方 README。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

import json
from src import data

annotations = data.load_annotations(data_root)
print('总页数:', len(annotations))

page0 = annotations[0]
print('顶层字段:', list(page0.keys()))
print('--- page_info ---')
print(json.dumps(page0['page_info'], ensure_ascii=False, indent=2))
print('--- 第一个 layout_det ---')
print(json.dumps(page0['layout_dets'][0], ensure_ascii=False, indent=2)[:1500])


## Step 3 — 可视化不同文档类型

每种文档类型抽 1 页展示标注：不同颜色对应不同 block 类别，数字是阅读顺序。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

import collections
from src import data
from src.visualization import draw_annotations
import matplotlib.pyplot as plt

annotations = data.load_annotations(data_root)
by_type = collections.defaultdict(list)
for p in annotations:
    by_type[data.page_attribute(p).get('data_source', 'unknown')].append(p)

for name in sorted(by_type):
    if name == 'unknown':
        continue
    page = by_type[name][0]
    img = data.load_page_image(data_root, page)
    fig = draw_annotations(img, page, title='document_type = %s (%d pages)' % (name, len(by_type[name])))
    display(fig)
    plt.close(fig)


## Step 4 — 统计分析

统计文档类型 / 语言 / 版面 / 子集分布，以及 block 类别与表格、公式数量。这些数字应与 `docs/notebook-design.md` §4.2 的官方审计数字一致。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data

annotations = data.load_annotations(data_root)
stats = data.build_stats(annotations)
print('总页数:', stats['pages'])
for key in ('document_type', 'language', 'layout', 'subset'):
    print(key, '=', stats[key])
print('table:', stats['table'])
print('formula:', stats['formula'])
print('relation:', stats['relation'])
top = sorted(stats['block_category'].items(), key=lambda kv: -kv[1])[:15]
print('block 类别 Top15:', top)


# What You Should Observe

- `subset` 字段把页面分成 v1.5 与三个困难子集——评测与教学子集划分要用它；
- 同一文档类型内部差异也可能很大（language / layout / watermark / fuzzy_scan 等属性）；
- 表格 665 个、公式相关 2523 个：一个 Overall 分数会把这些难度混在一起。


# Research Checkpoint

> **为什么说 Benchmark 不是「一个数字」，而是一组难度不同的数据？** 用你今天看到的至少两类分布（文档类型、困难子集）说明：一个平均分会隐藏什么问题。

**TODO：** 把答案写在 `results/nb01/research_checkpoint.md`。


# Exercises

1. **TODO：** 选一种文档类型（例如 exam_paper），统计它的语言与版面分布，并找出 2 页最难辨认的页面（利用 page_attribute 的 special_issue）；
2. **TODO：** 对比 `equation_hard` 与普通 v1.5 页面的平均公式数量，说明官方为什么把这 100 页单独标记；
3. **TODO：** 用 `data.sample_id()` 确认：同一 image_path 是否唯一对应一页？写一个断言验证。


# Takeaways

- OmniDocBench = 1651 页真实文档 + 28 类块级标注 + 属性标签 + 关系标注；
- 数据理解先于模型训练：分布、子集、属性决定后面如何切片分析；
- 官方数据仅研究用途，仓库只保存 revision 与下载方式，不提交数据本体。

**下一步**：[Notebook 04](04_Baseline_Inference.ipynb) — 建立 zero-shot 基线。
